# Muaalem FastConformer — live microphone streaming (Colab)

True cache-aware streaming inference for `obadx/muaalem-fastconformer-base-v1` from your **browser microphone**, via a Gradio app.

Unlike the file-based `infer_fastconformer_streaming` (which mel-transforms the whole file up front), this computes mel frames **incrementally** from the live audio buffer in a way that is bit-identical to the whole-file mel, and threads the encoder cache chunk by chunk (~0.5 s of new audio per step, ~0.5 s model lookahead).

**How to use:**
1. Runtime → Change runtime type → **GPU** (CPU works too, just slower).
2. Run all cells. The self-test cell verifies the streaming path against the trusted file simulation — expect `SELF TEST PASSED`.
3. Open the Gradio **share link**, allow microphone access, and speak. Stop recording to flush the final chunk.

In [ ]:
# Install dependencies (a few minutes — NeMo is heavy).
# librosa is pinned to the repo's locked version: the repo's inference.py does
# `from librosa.core import load`, which newer librosa (preinstalled on Colab)
# removed. If you change this cell after a first run, restart the runtime.
# If pip resolution fights Colab's preinstalled torch, retry without the
# nemo/transformers pins: only stable NeMo APIs are used.
!pip install -q "nemo_toolkit[asr]==2.1.0" "transformers>=5.9.0" "gradio>=6.20.0" "librosa==0.11.0" soxr

In [ ]:
# Get the modeling code. No `pip install -e`: the repo pins requires-python
# >=3.14 which Colab doesn't have — a plain sys.path insert works fine.
import os, sys

if not os.path.exists("/content/prepare-quran-dataset"):
    !git clone -b fastconformer --depth 1 https://github.com/Abdullah-Haytham/prepare-quran-dataset.git /content/prepare-quran-dataset

sys.path.insert(0, "/content/prepare-quran-dataset/src")
sys.path.insert(0, "/content/prepare-quran-dataset/tests")

In [ ]:
# HuggingFace auth for the model repo.
# Either store your token as a Colab secret named HF_TOKEN (key icon in the
# left sidebar), or you'll be prompted for it here.
# NOTE: the token must have READ access to obadx/muaalem-fastconformer-base-v1
# (a private repo returns 404 for tokens that cannot see it).
from huggingface_hub import login, whoami

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN").strip()
except Exception:
    from getpass import getpass
    HF_TOKEN = getpass("HuggingFace token: ").strip()

login(HF_TOKEN)
print("logged in as:", whoami(token=HF_TOKEN)["name"])

In [ ]:
# Load the model, processor, and vocab (token passed explicitly).
import torch
from streaming_mic_fastconformer import load_model_and_vocab, self_test, build_demo

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
model, vocab = load_model_and_vocab(
    "obadx/muaalem-fastconformer-base-v1", device, token=HF_TOKEN
)
print("levels:", list(vocab.keys()))

In [ ]:
# Correctness gate: feed a real recording through the true-streaming path in
# random-sized chunks and compare per-level argmax ids against the trusted
# file-based streaming simulation. Expect exact match on every level.
ok = self_test(
    "/content/prepare-quran-dataset/assets/audio-sampels/test_sample.mp3",
    model,
    device,
    vocab=vocab,
)
assert ok, "self test failed — do not trust the mic demo output"

In [ ]:
# Launch the mic app. Open the public share link, allow mic access, speak.
demo = build_demo(model, vocab, device)
demo.launch(share=True, debug=True)